## LeNet5

Outputs

- `models/lenet5.tflite` normal (INT8 input/output, per-tensor; MicroFlow-compatible)
- `models/lenet5_quantized.tflite` quantized (INT8 input/output, per-tensor; MicroFlow-compatible)
- `compiled_models/lenet5.mlir`
- `compiled_models/lenet5_quantized.mlir`

Keep `epochs` small if you only need artifacts quickly.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
COMPILED_DIR = REPO_ROOT / "compiled_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
COMPILED_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE = MODELS_DIR / "lenet5.tflite"
OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized.tflite"
OUT_MLIR = COMPILED_DIR / "lenet5.mlir"
OUT_MLIR_QUANT = COMPILED_DIR / "lenet5_quantized.mlir"

print("OUT_TFLITE:", OUT_TFLITE)
print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)
print("OUT_MLIR:", OUT_MLIR)
print("OUT_MLIR_QUANT:", OUT_MLIR_QUANT)

2026-03-27 11:05:11.179891: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 11:05:11.260992: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774605911.294597   35274 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774605911.304380   35274 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-27 11:05:11.383802: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

OUT_TFLITE: /home/nathan/Documents/ariel-os-tiny-ml/models/lenet5.tflite
OUT_TFLITE_QUANT: /home/nathan/Documents/ariel-os-tiny-ml/models/lenet5_quantized.tflite
OUT_MLIR: /home/nathan/Documents/ariel-os-tiny-ml/compiled_models/lenet5.mlir
OUT_MLIR_QUANT: /home/nathan/Documents/ariel-os-tiny-ml/compiled_models/lenet5_quantized.mlir


In [2]:
def build_lenet5() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(28, 28, 1), batch_size=1, name="input")
    x = tf.keras.layers.Conv2D(6, (5, 5), activation="relu", padding="valid")(inputs)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Conv2D(16, (5, 5), activation="relu", padding="valid")(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Reshape((256,), name="flatten_256")(x)
    x = tf.keras.layers.Dense(120, activation="relu", name="fc1")(x)
    x = tf.keras.layers.Dense(84, activation="relu", name="fc2")(x)
    logits = tf.keras.layers.Dense(10, name="fc3")(x)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="lenet5")

model = build_lenet5()
model.summary()

I0000 00:00:1774605913.366963   35274 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6153 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "lenet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 28, 28, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (1, 24, 24, 6)         │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (1, 12, 12, 6)         │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (1, 8, 8, 16)          │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (1, 4, 4, 16)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_256 (Reshape)           │ (1, 256)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (1, 120)               │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (1, 84)                │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (1, 10)                │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:

epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype(np.float32) / 255.0)[..., None]
x_test = (x_test.astype(np.float32) / 255.0)[..., None]

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2


I0000 00:00:1774605915.554789   35353 service.cc:148] XLA service 0x70a314005ad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774605915.554964   35353 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-27 11:05:15.584408: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774605915.707530   35353 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-03-27 11:05:16.244567: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_546', 8 bytes spill stores, 8 bytes spill loads

2026-03-27 11:05:16.420741: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_343', 1

422/422 - 10s - 25ms/step - accuracy: 0.8723 - loss: 0.4511 - val_accuracy: 0.9548 - val_loss: 0.1552
Epoch 2/2
422/422 - 1s - 2ms/step - accuracy: 0.9588 - loss: 0.1370 - val_accuracy: 0.9693 - val_loss: 0.1037
Test accuracy: 0.9656000137329102


In [ ]:
NORMAL_REP_SAMPLES = 50

def representative_data_gen_normal():
    n = min(NORMAL_REP_SAMPLES, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

int8_converter_normal = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter_normal.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter_normal.representative_dataset = representative_data_gen_normal
int8_converter_normal.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter_normal.target_spec.supported_types = [tf.int8]
int8_converter_normal.inference_input_type = tf.int8
int8_converter_normal.inference_output_type = tf.int8
int8_converter_normal.experimental_enable_resource_variables = True

for attr in (
    "_experimental_disable_per_channel",
    "_experimental_disable_per_channel_quantization",
):
    if hasattr(int8_converter_normal, attr):
        setattr(int8_converter_normal, attr, True)

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(int8_converter_normal, attr):
        setattr(int8_converter_normal, attr, False)

tflite_bytes = int8_converter_normal.convert()
OUT_TFLITE.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE, "bytes=", OUT_TFLITE.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE), experimental_delegates=[])
interp.allocate_tensors()
print("in ", interp.get_input_details()[0]["dtype"], interp.get_input_details()[0]["shape"])
print("out ", interp.get_output_details()[0]["dtype"], interp.get_output_details()[0]["shape"])

per_channel = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK")

try:
    ops = interp._get_ops_details()
    print("Ops:", [op.get("op_name") for op in ops])
except Exception as e:
    print("Couldn't list ops:", e)


Saved artifact at '/tmp/tmp3m4znuh8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  123852467904080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316824848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316827728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826192: TensorSpec(shape=(), dtype=tf.resource, name=None)
Wrote: /home/nathan/Documents/

/home/nathan/Documents/ariel-os-tiny-ml/building/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774605928.806266   35274 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774605928.806278   35274 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-27 11:05:28.806617: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp3m4znuh8
2026-03-27 11:05:28.806993: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-27 11:05:28.807003: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp3m4znuh8
I0000 00:00:1774605928.810291   35274 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-03-27 11:05:28.811013: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedMo

In [5]:
import shutil
from iree.compiler import tf as tfc

ARTIFACTS_DIR = REPO_ROOT / "build" / "mlir_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODEL_DIR = ARTIFACTS_DIR / "lenet5_saved_model"

shutil.rmtree(SAVED_MODEL_DIR, ignore_errors=True)
model.export(str(SAVED_MODEL_DIR))

mlir_bytes = tfc.compile_saved_model(
    str(SAVED_MODEL_DIR),
    import_only=True,
    exported_names=["serve"],
)
OUT_MLIR.write_bytes(mlir_bytes)

print("Wrote SavedModel:", SAVED_MODEL_DIR)
print("Wrote MLIR:", OUT_MLIR)
print("MLIR bytes:", OUT_MLIR.stat().st_size)


Saved artifact at '/home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/lenet5_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  123852467904080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316824848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316827728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826192: TensorSpec(s

2026-03-27 11:05:29.106152: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/lenet5_saved_model
2026-03-27 11:05:29.106734: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/lenet5_saved_model
2026-03-27 11:05:29.130668: W tensorflow/compiler/mlir/tensorflow/transforms/xla_validate_inputs.cc:96] missing entry functions


In [ ]:
import shutil

def representative_data_gen():
    for i in range(200):
        yield [x_train[i : i + 1].astype(np.float32)]

int8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_data_gen
int8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter.target_spec.supported_types = [tf.int8]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

for attr in (
    "_experimental_disable_per_channel",
    "_experimental_disable_per_channel_quantization",
):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, True)

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, False)

int8_tflite = int8_converter.convert()
OUT_TFLITE_QUANT.write_bytes(int8_tflite)

int8_interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
int8_interp.allocate_tensors()
in_info = int8_interp.get_input_details()[0]
out_info = int8_interp.get_output_details()[0]

print("wrote", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)
print("int8 input:", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("int8 output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])

per_channel = []
for td in int8_interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK (all tensors per-tensor or unquantized)")

shutil.copyfile(OUT_MLIR, OUT_MLIR_QUANT)


Saved artifact at '/tmp/tmpnvh9pthk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  123852467904080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316824848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316827728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316825040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123850316826192: TensorSpec(shape=(), dtype=tf.resource, name=None)
Wrote INT8 TFLite: /home/natha

/home/nathan/Documents/ariel-os-tiny-ml/building/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774605929.348934   35274 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774605929.348944   35274 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-27 11:05:29.349056: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpnvh9pthk
2026-03-27 11:05:29.349512: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-27 11:05:29.349523: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpnvh9pthk
2026-03-27 11:05:29.353269: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-27 11:05:29.375658: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on Save